# Validação 06 — Trechos rastreáveis

## Goal

Comprovar que o texto científico pode ser dividido por seção em trechos de tamanho controlado, com sobreposição, identificadores estáveis e proveniência completa.

## Setup

O notebook recupera novamente o artigo `PMC7805365` pelas fontes oficiais e aplica o módulo real de recorte. Nesta validação, cada trecho possui no máximo 120 palavras e 20 palavras de sobreposição.

In [1]:
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint
import os
import sys

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))

from fatofake import (
    ChunkingConfig,
    PmcClient,
    PubMedClient,
    chunk_article_content,
    prepare_search_plan,
    retrieve_article_content,
    search_pubmed,
    validate_analysis_input,
)

executed_at = datetime.now(timezone.utc).isoformat()
print(f"Execução UTC: {executed_at}")

Execução UTC: 2026-09-23T13:51:21.160618+00:00


## Steps

Recuperamos o conteúdo completo e recortamos cada seção isoladamente. Os intervalos `word_start` e `word_end` registram a posição dentro da seção original.

In [2]:
class PmidQueryPlanner:
    def generate_queries(self, claim: str) -> list[str]:
        return ["33431520[pmid]"]

analysis_input = validate_analysis_input(
    "O consumo de café altera o risco de câncer de próstata."
)
search_plan = prepare_search_plan(analysis_input, PmidQueryPlanner())
pubmed_result = search_pubmed(
    search_plan,
    PubMedClient(
        email=os.getenv("NCBI_EMAIL"),
        api_key=os.getenv("NCBI_API_KEY"),
    ),
    max_results_per_query=1,
)
content = retrieve_article_content(
    pubmed_result.publications[0],
    PmcClient(
        email=os.getenv("NCBI_EMAIL"),
        api_key=os.getenv("NCBI_API_KEY"),
    ),
)

In [3]:
chunking_config = ChunkingConfig(max_words=120, overlap_words=20)
chunks = chunk_article_content(content, chunking_config)

word_counts = [len(chunk.text.split()) for chunk in chunks]
summary = {
    "chunk_count": len(chunks),
    "section_count": len({chunk.section_index for chunk in chunks}),
    "minimum_words": min(word_counts),
    "maximum_words": max(word_counts),
    "chunks_by_section": dict(Counter(chunk.section for chunk in chunks)),
    "source_url": chunks[0].source_url,
}
pprint(summary)

{'chunk_count': 35,
 'chunks_by_section': {'Conclusions': 1,
                       'Discussion': 11,
                       'Introduction': 4,
                       'Methods': 10,
                       'Results': 9},
 'maximum_words': 120,
 'minimum_words': 25,
 'section_count': 5,
 'source_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7805365/'}


In [4]:
sample_chunk = chunks[0]
pprint({
    "chunk_id": sample_chunk.chunk_id,
    "pmid": sample_chunk.pmid,
    "pmcid": sample_chunk.pmcid,
    "section": sample_chunk.section,
    "chunk_index": sample_chunk.chunk_index,
    "word_interval": [sample_chunk.word_start, sample_chunk.word_end],
    "text_preview": sample_chunk.text[:500],
    "source_url": sample_chunk.source_url,
})

{'chunk_id': '33431520:1:1:2fd464f3b57fc0cd',
 'chunk_index': 1,
 'pmcid': 'PMC7805365',
 'pmid': '33431520',
 'section': 'Introduction',
 'source_url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7805365/',
 'text_preview': 'Prostate cancer is the second most frequently diagnosed '
                 'cancer and the sixth leading cause of cancer death in men. '
                 'There were 1 276 000 new cancer cases and 359 000 cancer '
                 'deaths in 2018.1 It is estimated that nearly three-quarters '
                 'of prostate cancer cases occur in developed countries.1 '
                 'Since the 1970s, the incidence of prostate cancer has also '
                 'increased rapidly in some Asian countries such as China, '
                 'Singapore and Japan, where the incidence has always been '
                 'much lower than in some Western countri',
 'word_interval': [0, 120]}


## Checks

As verificações confirmam tamanho máximo, sobreposição, unicidade, estabilidade e proveniência.

In [5]:
assert chunks
assert all(len(chunk.text.split()) <= chunking_config.max_words for chunk in chunks)
assert len({chunk.chunk_id for chunk in chunks}) == len(chunks)
assert all(chunk.pmid == "33431520" for chunk in chunks)
assert all(chunk.pmcid == "PMC7805365" for chunk in chunks)
assert all(chunk.source_kind == "PMC_FULL_TEXT" for chunk in chunks)
assert all(chunk.source_url == content.pmc_url for chunk in chunks)

chunks_by_section = defaultdict(list)
for chunk in chunks:
    chunks_by_section[chunk.section_index].append(chunk)
for section_chunks in chunks_by_section.values():
    for previous, current in zip(section_chunks, section_chunks[1:]):
        assert current.word_start == previous.word_end - chunking_config.overlap_words

second_run = chunk_article_content(content, chunking_config)
assert [chunk.chunk_id for chunk in chunks] == [chunk.chunk_id for chunk in second_run]

print(f"Validação aprovada para {len(chunks)} trechos rastreáveis.")

Validação aprovada para 35 trechos rastreáveis.


## Next Steps

O recorte estará validado quando todas as células forem executadas sem erros. A próxima etapa será recuperar e ordenar os trechos mais relevantes para uma alegação.